In [1]:
import pandas as pd
import matplotlib.pyplot as plt
# Use only Domestic/Men's/IPL/2022-2025 data

In [2]:
datas = [pd.read_csv(f'../data/Domestic/Men\'s/IPL/{y}/ipl_{y}_deliveries.csv') for y in range(2022, 2025)]
data = pd.concat(datas, ignore_index=True)
data.info()
data.sample(frac=1).reset_index(drop=True).head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 52351 entries, 0 to 52350
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   match_id          52351 non-null  int64  
 1   season            52351 non-null  int64  
 2   match_no          52351 non-null  int64  
 3   date              52351 non-null  object 
 4   venue             52351 non-null  object 
 5   batting_team      52351 non-null  object 
 6   bowling_team      52351 non-null  object 
 7   innings           52351 non-null  int64  
 8   over              52351 non-null  float64
 9   striker           52351 non-null  object 
 10  bowler            52351 non-null  object 
 11  runs_of_bat       52351 non-null  int64  
 12  extras            52351 non-null  int64  
 13  wide              52351 non-null  int64  
 14  legbyes           52351 non-null  int64  
 15  byes              52351 non-null  int64  
 16  noballs           52351 non-null  int64 

,match_id,season,match_no,date,venue,batting_team,bowling_team,innings,over,striker,bowler,runs_of_bat,extras,wide,legbyes,byes,noballs,wicket_type,player_dismissed,fielder
0,202324,2023,24,"Apr 17, 2023","M.Chinnaswamy Stadium, Bengaluru",RCB,CSK,2,19.3,Prabhudessai,Pathirana,6,0,0,0,0,0,NaN,NaN,NaN
1,202347,2023,47,"May 04, 2023","Rajiv Gandhi International Stadium, Hyderabad",KKR,SRH,1,2.5,Roy,Bhuvneshwar,0,0,0,0,0,0,NaN,NaN,NaN
2,202453,2024,53,"May 05, 2024","Himachal Pradesh Cricket Association Stadium, ...",CSK,PBKS,1,8.6,Jadeja,Harshal Patel,0,0,0,0,0,0,NaN,NaN,NaN
3,202265,2022,65,"May 17, 2022","Wankhede Stadium, Mumbai",SRH,MI,1,18.2,Washington Sundar,Meredith,3,0,0,0,0,0,NaN,NaN,NaN
4,202346,2023,46,"May 03, 2023","Punjab Cricket Association IS Bindra Stadium, ...",PBKS,MI,1,11.4,Livingstone,Chawla,1,0,0,0,0,0,NaN,NaN,NaN


In [8]:
# Further exploration of data
# Print unique values for each column
for column in data.columns:
    unique_values = data[column].unique()
    print(f"Column: {column}, Unique Values: {len(unique_values)}")
    if len(unique_values) <= 20:
        print(f"Values: {unique_values}")
    print("-" * 50)

unique_matches = data['match_id'].unique()
unique_teams = pd.concat([data['batting_team'], data['bowling_team']]).unique()
unique_players = pd.concat([data['striker'], data['bowler']]).unique()

Column: match_id, Unique Values: 217
--------------------------------------------------
Column: season, Unique Values: 3
Values: [2022 2023 2024]
--------------------------------------------------
Column: match_no, Unique Values: 74
--------------------------------------------------
Column: date, Unique Values: 177
--------------------------------------------------
Column: venue, Unique Values: 17
Values: ['Wankhede Stadium, Mumbai' 'Brabourne Stadium, Mumbai'
 'Dr DY Patil Sports Academy, Mumbai'
 'Maharashtra Cricket Association Stadium, Pune' 'Eden Gardens, Kolkata'
 'Narendra Modi Stadium, Ahmedabad'
 'Punjab Cricket Association IS Bindra Stadium, Mohali'
 'Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow'
 'Rajiv Gandhi International Stadium, Hyderabad'
 'M.Chinnaswamy Stadium, Bengaluru' 'MA Chidambaram Stadium, Chennai'
 'Arun Jaitley Stadium, Delhi' 'Barsapara Cricket Stadium, Guwahati'
 'Sawai Mansingh Stadium, Jaipur'
 'Himachal Pradesh Cricket Associatio

Now we'll create statistics for each player, including striker, bowler, fielder, and wicketkeeper stats. (a per-match basis)

The following are the columns we will include in our stats DataFrames:
#### Striker Stats:
- player: Name of the player
- match_id: Unique identifier for the match
- score: Total runs scored by the player in the match
- balls_faced: Total balls faced by the player in the match
- fours: Total number of fours hit by the player in the match
- sixes: Total number of sixes hit by the player in the match
- out: Whether the player was out in the match (True/False)

#### Bowler Stats:
- player: Name of the player
- match_id: Unique identifier for the match
- balls_bowled: Total balls bowled by the player in the match
- runs_conceded: Total runs conceded by the player in the match
- wickets: Total wickets taken by the player in the match

#### Fielder Stats:
- player: Name of the player
- match_id: Unique identifier for the match
- catches: Total number of catches taken by the player in the match
- run_outs: Total number of run outs effected by the player in the match

#### Wicketkeeper Stats:
- player: Name of the player
- match_id: Unique identifier for the match
- catches: Total number of catches taken by the wicketkeeper in the match
- run_outs: Total number of run outs effected by the wicketkeeper in the match

In [4]:
from collections import defaultdict

# dicts by player and match_id
striker_stats = defaultdict(lambda: defaultdict(lambda: {'score': 0, 'balls_faced': 0, 'fours': 0, 'sixes': 0, 'out': 0}))
bowler_stats = defaultdict(lambda: defaultdict(lambda: {'balls_bowled': 0, 'runs_conceded': 0, 'wickets': 0}))
fielder_stats = defaultdict(lambda: defaultdict(lambda: {'catches': 0, 'run_outs': 0, 'catches': 0, 'stumpings': 0}))

for delivery in data.itertuples():
    # Striker stats
    striker = delivery.striker
    match_id = delivery.match_id
    score = delivery.runs_of_bat
    balls_faced = 1
    fours = 1 if delivery.runs_of_bat == 4 else 0
    sixes = 1 if delivery.runs_of_bat == 6 else 0
    out = not pd.isna(delivery.player_dismissed)

    striker_stats[striker][match_id]['score'] += score
    striker_stats[striker][match_id]['balls_faced'] += balls_faced
    striker_stats[striker][match_id]['fours'] += fours
    striker_stats[striker][match_id]['sixes'] += sixes
    striker_stats[striker][match_id]['out'] += out

    # Bowler stats
    bowler = delivery.bowler
    balls_bowled = 1  # Each delivery is 1/6 of an over
    runs_conceded = delivery.runs_of_bat + delivery.extras
    wickets = 1 if delivery.player_dismissed and delivery.wicket_type not in ['hit wicket', 'retired out', 'obstructing the field'] else 0

    bowler_stats[bowler][match_id]['balls_bowled'] += balls_bowled
    bowler_stats[bowler][match_id]['runs_conceded'] += runs_conceded
    bowler_stats[bowler][match_id]['wickets'] += wickets

    # Fielder stats
    if getattr(delivery, "player_dismissed") and delivery.wicket_type in ['caught', 'run out']:
        fielder = delivery.fielder
        match delivery.wicket_type:
            case 'caught':
                fielder_stats[fielder][match_id]['catches'] += 1
            case 'run out':
                fielder_stats[fielder][match_id]['run_outs'] += 1
            case 'stumped':
                fielder_stats[fielder][match_id]['stumpings'] += 1

striker_stats_df = pd.DataFrame.from_dict({(i, j): striker_stats[i][j]
                            for i in striker_stats.keys() for j in striker_stats[i].keys()},
                        orient='index')
striker_stats_df.index.names = ['player', 'match_id']
striker_stats_df.reset_index(inplace=True)
bowler_stats_df = pd.DataFrame.from_dict({(i, j): bowler_stats[i][j]
                            for i in bowler_stats.keys() for j in bowler_stats[i].keys()},
                        orient='index')
bowler_stats_df.index.names = ['player', 'match_id']
bowler_stats_df.reset_index(inplace=True)
fielder_stats_df = pd.DataFrame.from_dict({(i, j): fielder_stats[i][j]
                            for i in fielder_stats.keys() for j in fielder_stats[i].keys()},
                        orient='index')
fielder_stats_df.index.names = ['player', 'match_id']
fielder_stats_df.reset_index(inplace=True)

N = 10
display(striker_stats_df.head(N))
display(bowler_stats_df.head(N))
display(fielder_stats_df.head(N))

,player,match_id,score,balls_faced,fours,sixes,out
0,Gaikwad,202201,0,5,0,0,1
1,Gaikwad,202207,1,4,0,0,1
2,Gaikwad,202211,1,4,0,0,1
3,Gaikwad,202217,16,13,3,0,1
4,Gaikwad,202222,17,16,3,0,1
5,Gaikwad,202229,73,49,5,5,1
6,Gaikwad,202233,0,1,0,0,1
7,Gaikwad,202238,30,28,4,0,1
8,Gaikwad,202246,99,59,6,6,1
9,Gaikwad,202249,28,23,3,1,1


,player,match_id,balls_bowled,runs_conceded,wickets
0,Umesh Yadav,202201,27,20,27
1,Umesh Yadav,202206,24,20,24
2,Umesh Yadav,202208,25,24,25
3,Umesh Yadav,202214,24,26,24
4,Umesh Yadav,202219,25,52,25
5,Umesh Yadav,202225,25,31,25
6,Umesh Yadav,202230,25,46,25
7,Umesh Yadav,202235,24,32,24
8,Umesh Yadav,202241,24,26,24
9,Umesh Yadav,202247,26,24,26


,player,match_id,catches,run_outs,stumpings
0,Nitish Rana,202201,1,0,0
1,Nitish Rana,202206,1,0,0
2,Nitish Rana,202208,2,0,0
3,Nitish Rana,202256,2,0,0
4,Nitish Rana,202302,1,0,0
5,Nitish Rana,202309,2,0,0
6,Shreyas Iyer,202201,1,0,0
7,Shreyas Iyer,202214,1,0,0
8,Shreyas Iyer,202253,2,0,0
9,Shreyas Iyer,202261,1,0,0


Now we aggregate the data to create these statistics for each player in total (including all matches). The following are the columns we will include in our aggregated stats DataFrames:
#### Striker Stats:
- player: Name of the player
- total_score: Total runs scored by the player across all matches
- total_balls_faced: Total balls faced by the player across all matches
- total_fours: Total number of fours hit by the player across all matches
- total_sixes: Total number of sixes hit by the player across all matches
- total_outs: Total number of times the player was out across all matches
- average_score: Average runs scored by the player per match
- matches_played: Total number of matches played by the player (where the player faced at least one ball)

#### Bowler Stats:
- player: Name of the player
- total_balls_bowled: Total balls bowled by the player across all matches
- total_runs_conceded: Total runs conceded by the player across all matches
- total_wickets: Total wickets taken by the player across all matches
- average_runs_conceded: Average runs conceded by the player per match
- matches_played: Total number of matches played by the player (where the player bowled at least one ball)

#### Fielder Stats:
- player: Name of the player
- total_catches: Total number of catches taken by the player across all matches
- total_run_outs: Total number of run outs effected by the player across all matches
- average_catches: Average number of catches taken by the player per match
- average_run_outs: Average number of run outs effected by the player per match

In [5]:
avg_striker_stats_df = striker_stats_df.groupby('player').agg({
    'score': 'sum',
    'balls_faced': 'sum',
    'fours': 'sum',
    'sixes': 'sum',
    'out': 'sum'
}).reset_index()
avg_striker_stats_df['average_score'] = avg_striker_stats_df['score'] / avg_striker_stats_df['balls_faced'].replace(0, 1)  # Avoid division by zero
avg_striker_stats_df['strike_rate'] = avg_striker_stats_df['average_score'] * 100  # Avoid division by zero
avg_striker_stats_df['matches_played'] = striker_stats_df.groupby('player')['match_id'].nunique().values
avg_striker_stats_df = avg_striker_stats_df.sort_values(by='average_score', ascending=False).reset_index(drop=True)

avg_bowler_stats_df = bowler_stats_df.groupby('player').agg({
    'balls_bowled': 'sum',
    'runs_conceded': 'sum',
    'wickets': 'sum'
}).reset_index()
avg_bowler_stats_df['economy_rate'] = (avg_bowler_stats_df['runs_conceded'] / avg_bowler_stats_df['balls_bowled'].replace(0, 1)) * 6  # Avoid division by zero
avg_bowler_stats_df['matches_played'] = bowler_stats_df.groupby('player')['match_id'].nunique().values
avg_bowler_stats_df = avg_bowler_stats_df.sort_values(by='economy_rate', ascending=True).reset_index(drop=True)

avg_fielder_stats_df = fielder_stats_df.groupby('player').agg({
    'catches': 'sum',
    'run_outs': 'sum',
    'stumpings': 'sum'
}).reset_index()
avg_fielder_stats_df['total_fielding_contributions'] = avg_fielder_stats_df['catches'] + avg_fielder_stats_df['run_outs'] + avg_fielder_stats_df['stumpings']

display(avg_striker_stats_df.head(N))
display(avg_bowler_stats_df.head(N))
display(avg_fielder_stats_df.head(N))

,player,score,balls_faced,fours,sixes,out,average_score,strike_rate,matches_played
0,Wood,9,3,0,1,0,3.000000,300.000000,1
1,Fraser-McGurk,330,149,32,28,8,2.214765,221.476510,9
2,Mark Wood,11,5,1,1,1,2.200000,220.000000,2
3,Sai Kishore,13,6,0,2,1,2.166667,216.666667,1
4,Wiese,21,11,0,3,1,1.909091,190.909091,3
5,Head,567,306,64,32,14,1.852941,185.294118,15
6,Glenn Phillips,39,22,2,4,5,1.772727,177.272727,5
7,Rashid Khan,323,184,22,25,13,1.755435,175.543478,25
8,Will Jacks,230,133,16,18,6,1.729323,172.932331,8
9,Philip Salt,627,370,72,32,20,1.694595,169.459459,20


,player,balls_bowled,runs_conceded,wickets,economy_rate,matches_played
0,Ravindra,12,7,12,3.500000,2
1,Praveen Dubey,19,19,19,6.000000,1
2,Short,25,25,25,6.000000,3
3,Maharaj,36,39,36,6.500000,2
4,Solanki,36,39,36,6.500000,2
5,Nehal Wadhera,12,13,12,6.500000,1
6,Jayant Yadav,24,26,24,6.500000,1
7,Glenn Phillips,18,20,18,6.666667,2
8,Narine,972,1084,972,6.691358,42
9,Bumrah,655,733,655,6.714504,27


,player,catches,run_outs,stumpings,total_fielding_contributions
0,(sub)Abhinav Manohar,1,0,0,1
1,(sub)Anuj Rawat,1,0,0,1
2,(sub)Anukul Roy,2,0,0,2
3,(sub)Brevis,1,0,0,1
4,(sub)Daniel Sams,1,0,0,1
5,(sub)Donovan Ferreira,1,0,0,1
6,(sub)Ferreira,1,0,0,1
7,(sub)Fraser-McGurk,2,0,0,2
8,(sub)Glenn Phillips,1,0,0,1
9,(sub)Gowtham,4,0,0,4


Combine everything into a per-match per-player aggregate dataset to use for XGBoost model training. The final dataset will include the following:
## Player and match columns
- **player**: Name of the player
- **match_id**: Unique identifier for the match
- **match_date**: Date of the match
- **player_roles**: List of roles the player can fill (e.g., `['striker', 'bowler', 'all-rounder', 'wicket-keeper']`) (note: all-rounder is a player who can both bat and bowl)
- **fantasy_points**: Running average of total fantasy points scored by the player in the match (calculated based on the scoring system)
- **team_played**: Name of the team the player played for in the match
- **team_faced**: Name of the team the player faced in the match
- **venue**: Name of the venue where the match was played
- **pitch**: Type of pitch (e.g., `green`, `flat`, `dust`, `slow`)
#### Weather stats
- **temperature**: Temperature in Celsius during the match
- **humidity**: Humidity percentage during the match
- **wind_speed**: Average wind speed in km/h during the match
<!--
>    ### Batting Stats
>    - **score**: Total runs scored by the player in the match
>    - **balls_faced**: Total balls faced by the player in the match
>    - **fours**: Total number of fours hit by the player in the match
>    - **sixes**: Total number of sixes hit by the player in the match
>    - **out**: Whether the player was out in the match (True/False)
>    ### Bowling stats
>    - **balls_bowled**: Total balls bowled by the player in the match
>    - **runs_conceded**: Total runs conceded by the player in the match
>    - **wickets**: Total wickets taken by the player in the match
>    ### Fielding stats
>    - **catches**: Total number of catches taken by the player in the match
>    - **run_outs**: Total number of run outs effected by the player in the match
-->
### Running average Stats
- **matches_played**: Total number of matches played
#### Batting stats
- **average_score**: Average score (per ball)
- **total_balls_faced**: Total number of balls faced
- **total_outs**: Total number of times player was out in matches
- **total_fours**: Total number of fours hit
- **total_sixes**: Total number of sixes hit
- **matches_played_batting**: Total number of matches played where the player batted
#### Bowling stats
- **average_runs_conceded**: Average runs conceded by the player per match
- **total_balls_bowled**: Total balls bowled by the player across all matches
- **total_wickets_taken**: Total wickets taken by the player across all matches
- **matches_played_bowling**: Total number of matches played by the player (where the player bowled at least one ball)
#### Fielding stats
- **average_catches**: Average number of catches taken by the player per match
- **average_run_outs**: Average number of run outs effected by the player per match
- **matches_played_fielding**: Total number of matches played by the player (where the player fielded at least once)


## Our model will be predicting per player per match fantasy scores

In [17]:
# Temporary df handling running averages
temp_df = defaultdict(lambda: defaultdict(lambda: {'average_score': 0, 'total_balls_faced': 0, 'total_outs': 0, 'total_fours': 0, 'total_sixes': 0, 'matches_played_batting': 0, 'average_runs_conceded': 0, 'total_balls_bowled': 0, 'total_wickets_taken': 0, 'matches_played_bowling': 0, 'average_catches': 0, 'average_run_outs': 0, 'matches_played_fielding': 0}))
# Get match dates in ascending order
for m in unique_matches:
    # get players that have played in this match
    date = data[data['match_id'] == m]['match_date'].iloc[0]
    players = data[data['match_id'] == m][['striker', 'bowler']].melt(value_name='player')['player'].unique()
    for p in players:
        # Calculate total stats for the player in this match
        score = data[(data['match_id'] == m) & (data['striker'] == p)]['runs_of_bat'].sum()
        balls_faced = data[(data['match_id'] == m) & (data['striker'] == p)].shape[0]
        fours = data[(data['match_id'] == m) & (data['striker'] == p) & (data['runs_of_bat'] == 4)].shape[0]
        sixes = data[(data['match_id'] == m) & (data['striker'] == p) & (data['runs_of_bat'] == 6)].shape[0]
        out = data[(data['match_id'] == m) & (data['striker'] == p)]['player_dismissed'].notna().sum()
        balls_bowled = data[(data['match_id'] == m) & (data['bowler'] == p)].shape[0]
        runs_conceded = data[(data['match_id'] == m) & (data['bowler'] == p)]['runs_of_bat'].sum() + data[(data['match_id'] == m) & (data['bowler'] == p)]['extras'].sum()
        wickets = data[(data['match_id'] == m) & (data['bowler'] == p)]['player_dismissed'].notna().sum()
        catches = data[(data['match_id'] == m) & (data['fielder'] == p) & (data['wicket_type'] == 'caught')].shape[0]
        run_outs = data[(data['match_id'] == m) & (data['fielder'] == p) & (data['wicket_type'] == 'run out')].shape[0]

        # Calculate total stats across all matches played by the player up to this match
        

    break

Match: 202201, Player: Gaikwad, Score: 0, Balls Faced: 5, Fours: 0, Sixes: 0, Out: 1, Balls Bowled: 0, Runs Conceded: 0, Wickets: 0, Catches: 0, Run Outs: 0
Match: 202201, Player: Uthappa, Score: 28, Balls Faced: 23, Fours: 2, Sixes: 2, Out: 1, Balls Bowled: 0, Runs Conceded: 0, Wickets: 0, Catches: 0, Run Outs: 0
Match: 202201, Player: Conway, Score: 3, Balls Faced: 8, Fours: 0, Sixes: 0, Out: 1, Balls Bowled: 0, Runs Conceded: 0, Wickets: 0, Catches: 0, Run Outs: 0
Match: 202201, Player: Rayudu, Score: 15, Balls Faced: 17, Fours: 1, Sixes: 1, Out: 0, Balls Bowled: 0, Runs Conceded: 0, Wickets: 0, Catches: 1, Run Outs: 0
Match: 202201, Player: Jadeja, Score: 26, Balls Faced: 29, Fours: 0, Sixes: 1, Out: 1, Balls Bowled: 24, Runs Conceded: 25, Wickets: 0, Catches: 1, Run Outs: 0
Match: 202201, Player: Shivam Dube, Score: 3, Balls Faced: 6, Fours: 0, Sixes: 0, Out: 1, Balls Bowled: 6, Runs Conceded: 15, Wickets: 0, Catches: 0, Run Outs: 0
Match: 202201, Player: Dhoni, Score: 50, Balls F

model will be trained on the following features:
- running_avg_fantasy_points
- team_played
- team_faced
- venue
- pitch
- temperature
- humidity
- wind_speed
- average_score
- total_balls_faced
- total_outs
- total_fours -> *might make more sense to use averages*
- total_sixes
- matches_played_batting
- average_runs_conceded
- total_balls_bowled  -> use avgs
- total_wickets_taken
- matches_played_bowling
- average_catches
- average_run_outs
- matches_played_fielding

and predicts `fantasy_points` for the player in that match